In [1]:
import numpy as np
import fhrr_driver as fd

In [2]:
A_space = fd.fhrr_space(100)
B_space = fd.fhrr_space(100)
E_space = fd.fhrr_space(100)
M_space = fd.fhrr_space(100000)
A_M_map = fd.fhrr_map(A_space, M_space)
B_M_map = fd.fhrr_map(B_space, M_space)
E_M_map = fd.fhrr_map(E_space, M_space)

In [3]:
A_R = A_space.init_random_vec()
A_G = A_space.init_random_vec()
A_B = A_space.init_random_vec()
B_R = B_space.init_random_vec()
B_G = B_space.init_random_vec()
B_B = B_space.init_random_vec()
A_vecs = [A_R, A_G, A_B]
B_vecs = [B_R, B_G, B_B]

In [4]:
def ball_game(p, n):
    p = p[:]
    event_outcomes = []
    for i in range(n):
        outcome = np.random.choice(len(p), p=np.array(p)/np.sum(p))
        event_outcomes.append(outcome)
        p[outcome] -= 1
    return event_outcomes

In [5]:
def query(memory_space, query_space, answer_space, event_space, QM_map, AM_map, EM_map, memory_vector, query_vector, answer_vectors):
    answer_matrix = np.concat(answer_vectors, axis=1)
    rel_events = EM_map.norm_backwards(memory_space.norm_unbind(memory_vector, QM_map.norm_forwards(query_vector)))
    rel_memory = memory_space.norm_unbind(memory_vector, EM_map.norm_forwards(rel_events))
    answer_vector = AM_map.backwards(rel_memory)
    pdf = answer_space.get_pdf(answer_vector, answer_matrix)
    return pdf

In [6]:
def recall(memory_space, answer_space, event_space, AM_map, EM_map, memory_vector, event_vector, answer_vectors):
    answer_matrix = np.concat(answer_vectors, axis=1)
    rel_memory = memory_space.norm_unbind(memory_vector, EM_map.norm_forwards(event_vector))
    answer_vector = AM_map.backwards(rel_memory)
    pdf = answer_space.get_pdf(answer_vector, answer_matrix)
    return pdf

In [7]:
memory_vector = M_space.init_zeros_vec()
memory_list = []
num_trials = 1000
p = [2, 5, 3]
n = 2
avg_outcome_a = [0, 0, 0]
avg_outcome_b = [0, 0, 0]
all_events = E_space.init_zeros_vec()
for trial in range(num_trials):
    outcome = ball_game(p, n)
    avg_outcome_a[outcome[0]] += 1/num_trials
    avg_outcome_b[outcome[1]] += 1/num_trials
    trial_event_vector = E_space.init_random_vec()
    all_events = E_space.bundle(all_events, trial_event_vector)
    A_M_vec = A_M_map.forwards_norm(A_vecs[outcome[0]])
    B_M_vec = B_M_map.forwards_norm(B_vecs[outcome[1]])
    E_M_vec = E_M_map.forwards_norm(trial_event_vector)
    outcome_vec = M_space.bundle(A_M_vec, B_M_vec)
    event_mem = M_space.bind(outcome_vec, E_M_vec)
    memory_vector = M_space.bundle(memory_vector*1, event_mem*0.001)
    memory_list.append(event_mem*0.001)
print(avg_outcome_a, avg_outcome_b)

[0.20600000000000016, 0.5100000000000003, 0.2840000000000002] [0.20800000000000016, 0.47700000000000037, 0.3150000000000002]


In [8]:
print(query(M_space, A_space, A_space, E_space, A_M_map, A_M_map, E_M_map, memory_vector, A_space.bundle(*A_vecs), A_vecs))

[[0.25086621 0.44368904 0.30544475]]


In [9]:
print(query(M_space, A_space, A_space, E_space, A_M_map, A_M_map, E_M_map, M_space.bundle(*memory_list), A_space.bundle(*A_vecs), A_vecs))

[[0.25086621 0.44368904 0.30544475]]


In [10]:
recall(M_space, A_space, E_space, A_M_map, E_M_map, memory_vector, E_space.normalize(all_events), A_vecs)

array([[0.22158022, 0.46828441, 0.31013537]])

In [11]:
print(query(M_space, A_space, A_space, E_space, A_M_map, A_M_map, E_M_map, memory_vector, A_R, A_vecs))

[[ 1.227743   -0.33931891  0.1115759 ]]


In [12]:
print(query(M_space, A_space, B_space, E_space, A_M_map, B_M_map, E_M_map, memory_vector, A_R, B_vecs))

[[0.14220401 0.48713148 0.37066451]]
